# Unified DriftSlice / FP32 → HWA → PCM Retention

Based on the uploaded DriftSlice v2.3 and AICAS native reproduction notebooks.

**Full run:** 30 digital FP32 epochs, 154 analog HWA epochs (`AnalogAdam`) on each method. Save *final* and *best-validation* models; use the latter for retention. FP32-Direct undergoes PCM retention without HWA.

**Important:** DriftSlice's per-device class-dependent conductance correction is an *idealized software-inference comparison*, not a demonstrated physical crossbar architecture. Training and retention are simulated in IBM AIHWKit. Full runs are computationally expensive.

This notebook imports the project-local `unified_experiment.py` rather than duplicating scientific code across notebooks. Start with smoke mode.

In [1]:
from pathlib import Path
import unified_experiment as experiment
import driftslice_core as ds
print('Available HWA methods:', experiment.ALL_TRAIN_METHODS)

Available HWA methods: ('PCM-HWA', 'PCM-DAR-HWA', 'PCM-DriftAttention-HWA', 'PCM-NSR-HWA', 'PCM-AttentionOnly-HWA', 'PCM-GainOnly-HWA')


## Configuration

Set `SMOKE=False` only after installing **AIHWKit + MNIST** and passing the smoke test. The full protocol requires CUDA by default. For early CPU testing, set `USE_CPU=True`. The full run is **30 + 154 epochs** per repetition, not an accelerated demonstration.

In [2]:
SMOKE = False
USE_CPU = False
MNIST_ROOT = 'data/MNIST/raw'
SELECTED_METHODS = experiment.ALL_TRAIN_METHODS
CONFIG = ds.Config(
    fast_mode=SMOKE,
    require_cuda=not USE_CPU,
    run_id='driftslice_v3_smoke' if SMOKE else 'driftslice_v3_full',
    methods=tuple(SELECTED_METHODS),
    data_root=MNIST_ROOT,
    allow_download=True,
)
print(CONFIG.resolved())

Config(seed=42, fast_mode=False, require_cuda=True, run_id='driftslice_v3_full', output_root='driftslice_verified_runs', resume_policy='auto_new', data_root='data/MNIST/raw', allow_download=True, training_semantics='fp32_pretrain_then_analog_hwa', digital_epochs=30, digital_lr=0.001, digital_max_lr=0.004, digital_min_val_accuracy=90.0, transfer_max_val_drop_pp=5.0, hwa_early_stop_drop_pp=8.0, hwa_early_stop_patience=2, batch_size=256, test_batch_size=512, epochs=154, train_repeats=5, lr=0.001, max_lr=0.004, weight_decay=0.0001, grad_clip=0.8, weight_scaling_omega=0.8, bits=8, weight_noise_std=0.02, output_noise_std=0.04, output_bound=10.0, input_range_init=3.0, gmax_us=25.0, ir_drop=0.0, dar_lambda_init=0.5, dar_fraction=0.8, dar_tc=0.01, dar_eta=1.0, dar_delta_mode='signed_initial', dar_g_units='siemens', dar_time_mode='literal_seconds', dar_times_s=(20.0, 60.0, 1800.0, 3600.0, 86400.0, 2592000.0, 15552000.0, 31557600.0, 63115200.0, 94672800.0, 157788000.0), realizations=30, retention

## Preflight checks and run

Execute the next cell to create a new run. The runner checks transferred weights and biases, checks training epoch counts, and selects the best **HWA epoch (1..154)** on validation. The DriftSlice number of bins is chosen using validation samples at 5 years; test data is only used after selection.

In [ ]:
# WARNING: This launches actual training and PCM simulations.
# Uncomment only after checking the configuration.
OUTPUT = experiment.run(CONFIG, Path('.').resolve())
print(OUTPUT)

{'torch': '2.6.0', 'aihwkit': '1.1.0', 'device': 'cuda', 'training': 'fp32_pretrain_then_analog_hwa'}
{'torch': '2.6.0', 'aihwkit': '1.1.0', 'device': 'cuda', 'training': 'fp32_pretrain_then_analog_hwa'}
PASS: AIHWKit non-Tensor state_dict checkpoint clone/save/load
PASS: FP32 digital-to-analog weight transfer, DAR gradient, snapshot forward, PCM timing and DriftSlice finite-state.
DAR reconstruction assumptions: {'eta': 1.0, 'delta_mode': 'signed_initial', 'g_units': 'siemens', 'time_mode': 'literal_seconds'}
First DAR-cycle lambda schedule:
 epoch_in_cycle         t_s  lambda_drift
              1        20.0  4.166667e-01
              2        60.0  1.104854e-01
              3      1800.0  5.064476e-03
              4      3600.0  1.689189e-03
              5     86400.0  5.170099e-05
              6   2592000.0  1.312476e-06
              7  15552000.0  1.735937e-07
              8  31557600.0  7.002122e-08
              9  63115200.0  2.934078e-08
             10  94672800.0  1.

## Results

Saved in `driftslice_verified_runs/<smoke|full>/<run_id>`:

- `digital_vs_hwa_test.csv` — clean FP32, direct analog, and HWA comparisons
- `best_validation_selection.csv` — selected HWA epoch after full 154-epoch training
- `driftslice_bin_ablation_validation.csv` — validation-only bin selection
- `retention_all_methods.csv` — matched time/realizations for each training + compensation method
- `retention_summary.csv` — mean errors and dispersion across training repetitions
- `retention_all_methods.png/pdf` — comparative retention curves

The original plotting scripts are archived under `legacy_reference`, because they use external paths and an older CSV layout.